In [18]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

print(f"TensorFlow Version: {tf.__version__}")
print(f"Devices: {tf.config.list_physical_devices()}")

# --- Environment Setup ---

# 1. Check for Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

MARKER_FILE = 'train_visual.py'

if IN_COLAB:
    print("\n--- Google Colab Detected ---")
    
    # Check if we are already in the root
    if os.path.exists(MARKER_FILE):
        print(f"Already in project root: {os.getcwd()}")
    
    # Check if we are in the inner 'ml_engine' package folder
    elif os.path.exists(os.path.join('..', MARKER_FILE)):
        os.chdir('..')
        print(f"Moved up to project root: {os.getcwd()}")
        
    else:
        # We need to get the code or move into it
        repo_name = 'ml_engine'
        
        if not os.path.exists(repo_name) and not os.path.exists(MARKER_FILE):
            print(f"Cloning {repo_name} repository...")
            !git clone https://github.com/Raynergy-svg/ml_engine.git
        
        if os.path.exists(repo_name):
            os.chdir(repo_name)
            print(f"Changed working directory to: {os.getcwd()}")

# 2. Check for Local Execution
else:
    print("\n--- Local Environment Detected ---")
    if os.path.exists(MARKER_FILE):
        pass
    elif os.path.exists('/Users/mirelacertan/Documents/ml_engine'):
        os.chdir('/Users/mirelacertan/Documents/ml_engine')
        print(f"Changed working directory to: {os.getcwd()}")
    elif os.path.exists(os.path.join('..', MARKER_FILE)):
        os.chdir('..')

# 3. Final Verification
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

if os.path.exists(MARKER_FILE):
    print("\nSUCCESS: Project root configured correctly.")
    print(f"Working Directory: {os.getcwd()}")
else:
    print("\nCRITICAL ERROR: Could not find project files.")
    print(f"Current Directory: {os.getcwd()}")
    print("Contents:", os.listdir('.'))

In [ ]:
# GPU setup (Colab-friendly): verify GPU, enable memory growth, and set mixed precision.
import importlib
import sys
import tensorflow as tf

# Optional: install missing non-TensorFlow deps on Colab (do NOT reinstall TensorFlow here).
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    wanted = [
        ("python-dotenv", "dotenv"),
        ("pandas", "pandas"),
        ("numpy", "numpy"),
        ("scikit-learn", "sklearn"),
        ("matplotlib", "matplotlib"),
        ("tqdm", "tqdm"),
        ("pyyaml", "yaml"),
        ("rich", "rich"),
        ("typer", "typer"),
        ("plotly", "plotly"),
    ]
    missing = []
    for pkg, mod in wanted:
        try:
            importlib.import_module(mod)
        except Exception:
            missing.append(pkg)

    if missing:
        print("Installing missing packages:", missing)
        !{sys.executable} -m pip install -q {' '.join(missing)}

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

if not gpus:
    print("No GPU detected.")
    if IN_COLAB:
        print("Colab: Runtime → Change runtime type → Hardware accelerator → GPU")
else:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as e:
            print("Memory growth not set:", e)

    try:
        tf.keras.mixed_precision.set_global_policy("mixed_float16")
        print("Mixed precision policy:", tf.keras.mixed_precision.global_policy())
    except Exception as e:
        print("Could not enable mixed precision:", e)


In [38]:
# Configuration
CONFIG = {
    'model_type': 'tft',
    'multi_task': True,
    'epochs': 100,
    'batch_size': 256,  # Increased from 64 to 256 for faster GPU training
    'ensemble_size': 3,
    'fetch_data': True,
    'data_dir': 'trained_data/data',
    'checkpoint_dir': 'trained_data/checkpoints/tensorflow',
    'log_dir': 'trained_data/tensorboard',
    'instruments': ['USD_JPY', 'EUR_USD', 'GBP_USD'],
    'candles': 5000
}

print("Configuration loaded successfully.")
print(f"Model: {CONFIG['model_type'].upper()}")
print(f"Batch Size: {CONFIG['batch_size']}")
print(f"Instruments: {CONFIG['instruments']}")
print("You can now run the next cell to fetch data.")

In [39]:
# Fetch Fresh Data (Optional)
import sys
import os
from dotenv import load_dotenv

# Ensure project root is in path
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# Load environment variables from .env file
if os.path.exists('.env'):
    load_dotenv('.env')
    print("Loaded configuration from .env file.")
else:
    print("Warning: .env file not found. OANDA credentials might be missing.")

try:
    from train_visual import fetch_fresh_oanda_data
except ImportError as e:
    print(f"Error importing train_visual: {e}")
    raise

if 'CONFIG' not in locals():
    print("Error: CONFIG is not defined. Please run the 'Configuration' cell above first.")
else:
    if CONFIG['fetch_data']:
        print("Fetching fresh data from OANDA...")
        try:
            fetch_fresh_oanda_data(
                instruments=CONFIG['instruments'],
                count=CONFIG['candles'],
                output_dir=CONFIG['data_dir']
            )
            print("Data fetch complete.")
        except Exception as e:
            print(f"Warning: Could not fetch data ({e}).")
            print("Using existing files in 'trained_data/data' instead.")
            print("You can proceed to the next cell if you have previously downloaded data.")

In [40]:
# Load Data (Patched for Colab/Remote Execution)
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Any
from data_processing import prepare_sequences
from multitask_labels import split_time_series
from feature_engineering import FeatureEngineering
from utils import load_config

# --- HOTFIX: Define patched functions locally ---
# This ensures the 'direction' fix works even if the remote files are stale.

@dataclass(frozen=True)
class MultitaskTargets:
    trend: np.ndarray
    direction: np.ndarray
    risk: np.ndarray
    state: np.ndarray

def _safe_close_series(df: pd.DataFrame, close_col: str) -> np.ndarray:
    if close_col not in df.columns:
        raise ValueError(f"Missing close column: {close_col}")
    close = pd.to_numeric(df[close_col], errors="coerce").to_numpy(dtype=float)
    if np.isnan(close).any():
        close = np.nan_to_num(close, nan=np.nanmedian(close))
    return close

def build_multitask_targets_patched(
    df,
    sequence_length,
    target_shift,
    close_col="close",
    risk_window=14,
    state_classes=3,
):
    close = _safe_close_series(df, close_col)

    # 1-step returns (used for risk calculation)
    returns = np.zeros_like(close, dtype=float)
    returns[1:] = (close[1:] / np.clip(close[:-1], 1e-12, None)) - 1.0

    # Risk (rolling std of returns)
    risk = np.zeros_like(returns, dtype=float)
    for i in range(len(returns)):
        start = max(0, i - risk_window + 1)
        window = returns[start : i + 1]
        risk[i] = float(np.std(window))

    # State (risk quantiles)
    qs = np.linspace(0.0, 1.0, state_classes + 1)
    edges = np.unique(np.quantile(risk, qs))
    if len(edges) < 3:
        state = np.zeros_like(risk, dtype=np.int64)
    else:
        state = np.digitize(risk, edges[1:-1], right=False).astype(np.int64)

    # Align to sequences (horizon return from end-of-seq to +target_shift)
    n = len(close) - sequence_length - target_shift + 1
    if n <= 0:
        raise ValueError("Not enough rows for given sequence_length and target_shift")

    base_indices = np.arange(n, dtype=int) + sequence_length - 1
    future_indices = base_indices + target_shift

    horizon_ret = (close[future_indices] / np.clip(close[base_indices], 1e-12, None)) - 1.0

    trend_aligned = horizon_ret.astype(np.float32)
    direction_aligned = (horizon_ret > 0).astype(np.float32)
    risk_aligned = risk[future_indices].astype(np.float32)
    state_aligned = state[future_indices].astype(np.int64)

    # Normalize risk to [0,1]
    r_min, r_max = float(np.min(risk_aligned)), float(np.max(risk_aligned))
    risk_norm = (risk_aligned - r_min) / (r_max - r_min) if r_max > r_min else np.zeros_like(risk_aligned)

    return MultitaskTargets(
        trend=trend_aligned,
        direction=direction_aligned,
        risk=risk_norm.astype(np.float32),
        state=state_aligned,
    )

def load_data_patched(csv_path, config, state_classes=3, validation_split=0.2):
    print(f"Loading {csv_path}...")
    df = pd.read_csv(csv_path)
    df["time"] = pd.to_datetime(df["time"])
    df.set_index("time", inplace=True)

    # Features
    fe = FeatureEngineering()
    df = fe.create_features(df)

    # Sequences
    seq_len = config.get("model", {}).get("sequence_length", 60)
    target_shift = 1
    feature_cols = [c for c in df.columns if c not in ["open", "high", "low", "close", "volume"]]

    x, y_price, meta = prepare_sequences(
        df=df,
        sequence_length=seq_len,
        target_column="close",
        feature_columns=feature_cols,
        target_shift=target_shift,
        scale_features=True,
        scale_target=True,
    )

    # Targets
    targets = build_multitask_targets_patched(df, seq_len, target_shift, state_classes=state_classes)

    # Split
    train_idx, val_idx = split_time_series(len(x), val_fraction=validation_split)

    def _col(xarr):
        return np.asarray(xarr, dtype=np.float32).reshape(-1, 1)

    def create_dict(idx):
        return {
            "price": _col(y_price[idx]),
            "trend": _col(targets.trend[idx]),
            "direction": _col(targets.direction[idx]),
            "risk": _col(targets.risk[idx]),
            "state_logits": np.eye(state_classes)[targets.state[idx]].astype(np.float32),
        }

    return (
        x[train_idx].astype(np.float32),
        create_dict(train_idx),
        x[val_idx].astype(np.float32),
        create_dict(val_idx),
        None,
        None,
        meta,
    )

# --- END HOTFIX ---

# Find the main data file
data_dir = Path(CONFIG["data_dir"])
data_files = list(data_dir.glob("*.csv"))
if not data_files:
    raise FileNotFoundError(f"No CSV files found in {data_dir}")

target_file = next((f for f in data_files if "USD_JPY" in f.name), data_files[0])
print(f"Training on: {target_file}")

project_config = load_config("config.yaml")

# Use the PATCHED loader
x_train, y_train, x_val, y_val, _, _, meta = load_data_patched(
    str(target_file),
    project_config,
    state_classes=3,
    validation_split=0.2,
)

print(f"Training Samples: {len(x_train)}")
print(f"Validation Samples: {len(x_val)}")
print(f"Input Shape: {x_train.shape}")
print(f"Target Keys: {list(y_train.keys())}")
print("Data loaded successfully with direction labels (patched).")


In [46]:
# Clear Session
import tensorflow as tf
tf.keras.backend.clear_session()
print("Session cleared. Ready to rebuild model.")

In [47]:
# Build Model (GPU-ready)
from tensorflow_engine import TensorFlowEngine
import numpy as np
import tensorflow as tf

# Construct Engine Config
tf_config = {
    'model': {
        'type': CONFIG['model_type'],
        'input_size': x_train.shape[-1],
        'hidden_size': 128,
        'num_layers': 3,
        'dropout': 0.35,
        'num_heads': 4,
        'multi_task': CONFIG['multi_task'],
        'state_classes': 3,
        'sequence_length': x_train.shape[1],
    },
    'optimizer': {
        'type': 'adamw',
        'learning_rate': 3e-4,
        'weight_decay': 0.001,
        'clipnorm': 1.0,
    },
    'training': {
        'epochs': CONFIG['epochs'],
        'early_stopping_patience': 25,
        # On GPU we want graph mode; on CPU the engine auto-enables eager for TFT.
        'run_eagerly': False,
    },
    'paths': {
        'checkpoint_dir': CONFIG['checkpoint_dir'],
        'tensorboard_dir': CONFIG['log_dir'],
    },
    'mixed_precision': True,
    'batch_size': CONFIG['batch_size'],
    'adaptive_loss_weights': True,
    'unified_head_loss_weights': {
        'price': 1.0,
        'trend': 0.5,
        'direction': 20.0,
        'risk': 2.0,
        'state_logits': 5.0,
    },
}

# IMPORTANT: we compile before engine.train(), so set direction loss here.
try:
    y_dir = np.array(y_train['direction']).reshape(-1)
    up_rate = float(np.mean(y_dir >= 0.5)) if y_dir.size else 0.5
    tf_config['direction_loss'] = 'bce' if abs(up_rate - 0.5) <= 0.05 else 'focal'
    tf_config['focal_alpha'] = float(1.0 - up_rate)
    print(f"Direction up_rate={up_rate:.3f} → direction_loss={tf_config['direction_loss']}")
except Exception as e:
    print("Could not infer direction balance:", e)

# Initialize Engine
engine = TensorFlowEngine(tf_config)
engine.build_model()

# Explicitly build the model by running a dummy forward pass
print("Initializing model weights...")
dummy_input = tf.zeros((1, x_train.shape[1], x_train.shape[-1]))
_ = engine.model(dummy_input)

# Show Model Summary
engine.model.summary()

### GPU + Direction-First Configuration
This notebook is set up to train TFT on GPU (when available) and to avoid "coin-flip" direction learning.

Key settings used in the model build cell:
- Mixed precision on GPU (`mixed_float16`)
- Direction loss: auto-select BCE when labels are ~50/50
- Loss weights prioritize direction: `direction=20.0`, reduce easier heads (`state_logits=5.0`, `risk=2.0`)
- Adaptive loss weights enabled (now includes `direction`)

In [48]:
# Train
import tensorflow as tf

print("--- GPU Diagnostics ---")
gpu_name = tf.test.gpu_device_name()
if gpu_name:
    print(f"GPU Found: {gpu_name}")
    try:
        print("GPU Status:")
        !nvidia-smi
    except Exception:
        print("Could not run nvidia-smi (might be non-NVIDIA GPU)")
else:
    print("No GPU detected; training will be slower.")

print("\n--- Mixed Precision Check ---")
try:
    policy = tf.keras.mixed_precision.global_policy()
    print(f"Global Policy: {policy.name}")
    print(f"Compute Dtype: {policy.compute_dtype}")
    print(f"Variable Dtype: {policy.variable_dtype}")
except Exception:
    print("Could not check mixed precision policy.")

print("\n--- Starting Training ---")
history = engine.train(x_train, y_train, x_val, y_val)


In [44]:
# Save Scalers for Buddy
from tensorflow_data_pipeline import save_scalers

# Save scalers so Buddy can normalize live data correctly
save_scalers(
    {
        'scalers': {
            'feature_scaler': meta.get('feature_scaler'),
            'target_scaler': meta.get('target_scaler')
        },
        'n_features': x_train.shape[-1],
        'sequence_length': x_train.shape[1],
        'state_classes': 3
    },
    CONFIG['checkpoint_dir']
)
print(f"Scalers saved to {CONFIG['checkpoint_dir']}")

In [51]:
# Visualize Training Results
import matplotlib.pyplot as plt

def plot_training_history(history):
    """Plot training metrics from the history object."""
    metrics = ["loss", "direction_dir_acc", "price_mae"]
    titles = ["Total Loss", "Direction Accuracy (Up/Down)", "Price MAE"]

    plt.figure(figsize=(18, 5))

    for i, metric in enumerate(metrics):
        if metric not in history.history:
            continue

        plt.subplot(1, 3, i + 1)
        plt.plot(history.history[metric], label=f"Train {metric}")
        if f"val_{metric}" in history.history:
            plt.plot(history.history[f"val_{metric}"], label=f"Val {metric}")

        plt.title(titles[i])
        plt.xlabel("Epoch")
        plt.ylabel("Value")
        plt.legend()
        plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

if "history" in locals():
    plot_training_history(history)
else:
    print("No training history found. Run the 'Train' cell first.")


In [49]:
# 🚀 Launch TensorBoard
# This allows you to explore the logs interactively
%load_ext tensorboard
%tensorboard --logdir trained_data/tensorboard